# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding 1: 'Content refreshes recover 40% of lost traffic within 30 days.'**
- **My methodology question:** Where does the label come from? If the label is just an 'observed traffic increase', how is the model isolating the effect of the refresh from normal seasonal bounce-backs? Does the validation design use a control group or quasi-experimental design, or is it just a naive before-and-after observation?

**Finding 2: 'AI-generated content decays 3x faster than human content.'**
- **My methodology question:** How is the 'AI-generated' label defined? If the label relies on a third-party AI-detector tool, could the model simply be learning the signature of the detector rather than true decay? Does the validation design control for the domain authority or age of the content across the two groups?

## 2. My model under an honest split (before/after)

Here, I re-run my model from Week 5 to show the difference between a "Random Split" (which allows the model to cheat by memorizing the clients) and an "Honest Split" (grouped by client, which forces the model to generalize to unseen domains).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

features = ['content_age_days', 'word_count', 'impressions_90d', 'clicks_90d', 'avg_position', 'ctr', 'engagement_rate']
X = df[features + ['client_id']]
y = df['is_declining_label']

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

rf_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42))
])

# ---------------------------------------------------------
# 1. Random Split (Dishonest)
# ---------------------------------------------------------
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.25, random_state=42)
rf_pipe.fit(X_train_r[features], y_train_r)
prec_random = precision_at_k(rf_pipe.predict_proba(X_test_r[features])[:, 1], y_test_r)

# ---------------------------------------------------------
# 2. Grouped Split (Honest)
# ---------------------------------------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=X['client_id']))
X_train_h, X_test_h = X.iloc[train_idx], X.iloc[test_idx]
y_train_h, y_test_h = y.iloc[train_idx], y.iloc[test_idx]

rf_pipe.fit(X_train_h[features], y_train_h)
prec_honest = precision_at_k(rf_pipe.predict_proba(X_test_h[features])[:, 1], y_test_h)

print(f"Random Split Precision@50: {prec_random:.3f}")
print(f"Honest (Grouped) Split Precision@50: {prec_honest:.3f}")
print(f"Gap (Memorization penalty): {prec_random - prec_honest:.3f}")

Random Split Precision@50: 0.900
Honest (Grouped) Split Precision@50: 0.660
Gap (Memorization penalty): 0.240


## 3. Leakage audit

I will intentionally inject `trend_pct` into the features to prove that my test harness catches label leakage (since `trend_pct` was mathematically used to compute the label).

In [2]:
# Add leaky feature
leaky_features = features + ['trend_pct']

# Train WITH leak
X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(df[leaky_features], y, test_size=0.25, random_state=42)
rf_pipe.fit(X_train_leak, y_train_leak)
auc_leak = roc_auc_score(y_test_leak, rf_pipe.predict_proba(X_test_leak)[:, 1])

print(f"ROC-AUC WITH leaked feature ('trend_pct'): {auc_leak:.3f} (Suspiciously perfect!)")

# Train WITHOUT leak
rf_pipe.fit(X_train_r[features], y_train_r)
auc_clean = roc_auc_score(y_test_r, rf_pipe.predict_proba(X_test_r[features])[:, 1])

print(f"ROC-AUC WITHOUT leaked feature: {auc_clean:.3f} (Honest)")

ROC-AUC WITH leaked feature ('trend_pct'): 0.999 (Suspiciously perfect!)
ROC-AUC WITHOUT leaked feature: 0.726 (Honest)


## 4. Claim rewrite

**Original Bold Claim:** *"My model predicts which pages will decline, proving that age causes traffic decay and refreshing them will instantly save the traffic."*

**Rewritten Safe Claim:** *"The model **observes** historical patterns to flag pages with a high statistical risk of **directional** decline. It acts as **decision-support** for content teams, though it cannot prove that age is the causal factor of the decay, nor can it guarantee a refresh will stop the decline."*

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.